# Veritas PoC on Google Colab

1. `Runtime > Change runtime type > T4 GPU > Save`
2. **`Runtime > Restart session`** (required — CUDA is not visible until you restart after switching to GPU)
3. Run cells **in order**, one at a time. Do not run another cell while one is executing.

Colab may warn *"connected to GPU but not utilizing it"* during **pip install / git clone** — that is normal. It should go away once cell 6 runs and prints `Veritas selected device: cuda`.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'CUDA not visible. Set runtime to T4 GPU, then Runtime > Restart session, '
    'then re-run from the top.'
)
print(torch.cuda.get_device_name(0))
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import os
if os.path.basename(os.getcwd()) != 'algoverse-PoC':
    if not os.path.isdir('algoverse-PoC'):
        !git clone https://github.com/abdelmagid07/algoverse-PoC.git
    %cd algoverse-PoC
!git pull --ff-only

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token: '))

In [ ]:
import os
os.environ['VERITAS_DEVICE'] = 'cuda'  # belt-and-suspenders on Colab

from pathlib import Path
for name, p in [
    ('trajectories', 'results/trajectories.json'),
    ('fast scores', 'results/fast_scores.json'),
    ('summary', 'results/poc_summary.md'),
]:
    path = Path(p)
    print(f"{'OK' if path.exists() else 'MISSING':7} {name}")

In [ ]:
# Whole pipeline (~15-30 min). Wait for "=== Done ===".
# If interrupted, re-run THIS cell only — it resumes from checkpoints.
!python run_pipeline.py

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

for p in ['results/correlation.json', 'results/importance_by_step.png', 'results/poc_summary.md']:
    assert Path(p).exists(), f'Missing {p} — run cell 6 to completion'

print(json.dumps(json.load(open('results/correlation.json')), indent=2))
display(Image('results/importance_by_step.png'))
display(Markdown(open('results/poc_summary.md').read()))